# Validate Knowledge

Validates integrated Work knowledge candidates before promotion to Gold. The notebook verifies required identifiers, classification, source context, content, and canonical ID uniqueness, reconciles candidate and validated row counts, and promotes only trusted knowledge items to the Gold layer.

## Input Parameters

In [1]:
p_execution_step_id = 0
p_created_by = ""

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 3, Finished, Available, Finished, False)

## Imports

In [2]:
import json
from datetime import datetime, timezone

from pyspark.sql import functions as F

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 4, Finished, Available, Finished, False)

## Helper Functions

Defines reusable helpers for combining validation flags and recording validation results.

In [ ]:
def get_validation_condition(df):
    """
    Returns a Boolean Spark condition that is True only when all
    temporary validation flags in the DataFrame have passed.
    """

    validation_flag_columns = [
        column_name
        for column_name in df.columns
        if column_name.startswith("is_")
        and column_name.endswith("_passed")
    ]

    condition = F.lit(True)

    for column_name in validation_flag_columns:
        condition = condition & F.col(column_name)

    return condition

In [ ]:
def add_validation_result(
    validation_results,
    details,
    affected_row_count,
    validation_code
):
    """
    Appends a validation result to the validation result collection.
    """

    validation_results.append({
        "details": details,
        "affected_row_count": affected_row_count,
        "validation_code": validation_code
    })

## Load Work Knowledge

Loads the complete current integrated knowledge candidate set from the Work layer for validation before Gold promotion.

In [5]:
def load_work_knowledge():
    """
    Loads the current integrated knowledge candidates
    from the Work layer.
    """

    return spark.table("work.knowledge_items")

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 7, Finished, Available, Finished, False)

## Validate Knowledge Items

Applies row-level validation rules to integrated Work knowledge candidates before Gold promotion.

In [ ]:
def validate_required_knowledge_id(
    work_knowledge_items,
    validation_results
):
    """
    Validates that every knowledge item has a canonical knowledge_id.
    """

    work_knowledge_items = (
        work_knowledge_items
        .withColumn(
            "is_knowledge_id_passed",
            F.col("knowledge_id").isNotNull()
        )
    )

    affected_row_count = (
        work_knowledge_items
        .filter(~F.col("is_knowledge_id_passed"))
        .count()
    )

    add_validation_result(
        validation_results=validation_results,
        details=(
            "Required ID validation: "
            "knowledge_items.knowledge_id contains null values."
        ),
        affected_row_count=affected_row_count,
        validation_code="NOT_NULL"
    )

    return work_knowledge_items

In [ ]:
def validate_required_content(
    work_knowledge_items,
    validation_results
):
    """
    Validates that every knowledge item contains usable content.
    """

    work_knowledge_items = (
        work_knowledge_items
        .withColumn(
            "is_content_passed",
            (
                F.col("content").isNotNull() &
                (F.length(F.trim(F.col("content"))) > 0)
            )
        )
    )

    affected_row_count = (
        work_knowledge_items
        .filter(~F.col("is_content_passed"))
        .count()
    )

    add_validation_result(
        validation_results=validation_results,
        details=(
            "Required content validation: "
            "knowledge_items.content contains null or empty values."
        ),
        affected_row_count=affected_row_count,
        validation_code="NOT_NULL"
    )

    return work_knowledge_items

In [ ]:
def validate_required_classification(
    work_knowledge_items,
    validation_results
):
    """
    Validates that every knowledge item contains the required
    classification attributes.
    """

    required_classification_columns = [
        "knowledge_type",
        "source_type"
    ]

    for column_name in required_classification_columns:

        flag_column = f"is_{column_name}_passed"

        work_knowledge_items = (
            work_knowledge_items
            .withColumn(
                flag_column,
                (
                    F.col(column_name).isNotNull() &
                    (F.length(F.trim(F.col(column_name))) > 0)
                )
            )
        )

        affected_row_count = (
            work_knowledge_items
            .filter(~F.col(flag_column))
            .count()
        )

        add_validation_result(
            validation_results=validation_results,
            details=(
                "Required classification validation: "
                f"knowledge_items.{column_name} contains null or empty values."
            ),
            affected_row_count=affected_row_count,
            validation_code="NOT_NULL"
        )

    return work_knowledge_items

In [ ]:
def validate_knowledge_id_uniqueness(
    work_knowledge_items,
    validation_results
):
    """
    Validates that knowledge_id uniquely identifies each knowledge item.
    """

    duplicate_knowledge_ids = (
        work_knowledge_items
        .groupBy("knowledge_id")
        .count()
        .filter(F.col("count") > 1)
        .select("knowledge_id")
    )

    work_knowledge_items = (
        work_knowledge_items.alias("k")
        .join(
            duplicate_knowledge_ids.alias("d"),
            F.col("k.knowledge_id") == F.col("d.knowledge_id"),
            "left"
        )
        .withColumn(
            "is_knowledge_id_unique_passed",
            F.col("d.knowledge_id").isNull()
        )
        .select(
            "k.*",
            "is_knowledge_id_unique_passed"
        )
    )

    affected_row_count = (
        work_knowledge_items
        .filter(~F.col("is_knowledge_id_unique_passed"))
        .count()
    )

    add_validation_result(
        validation_results=validation_results,
        details=(
            "Uniqueness validation: duplicate knowledge_id values "
            "found in knowledge_items."
        ),
        affected_row_count=affected_row_count,
        validation_code="DUPLICATE"
    )

    return work_knowledge_items

## Determine Validation Outcome

Combines all row-level validation flags into a final result and returns the knowledge items that passed every validation rule.

In [ ]:
def determine_validation_outcome(
    work_knowledge_items
):
    """
    Combines all validation flags and returns only knowledge items
    that passed every validation rule.
    """

    work_knowledge_items = (
        work_knowledge_items
        .withColumn(
            "is_validation_passed",
            get_validation_condition(work_knowledge_items)
        )
    )

    valid_knowledge_items = (
        work_knowledge_items
        .filter(F.col("is_validation_passed"))
    )

    return (
        work_knowledge_items,
        valid_knowledge_items
    )

## Reconcile Row Counts

Compares the complete current Work knowledge candidate count with the number of validated records eligible for Gold promotion.

In [13]:
def reconcile_knowledge_row_counts(
    work_knowledge_items,
    valid_knowledge_items
):
    """
    Reconciles Work knowledge candidates against validated
    knowledge items prepared for Gold promotion.
    """

    source_row_count = work_knowledge_items.count()
    target_row_count = valid_knowledge_items.count()

    reconciliation_results = [
        {
            "metric_name": "ROW_COUNT",
            "reconciliation_key": "KNOWLEDGE_ITEMS",
            "reconciliation_side": "SOURCE",
            "metric_value_bigint": source_row_count
        },
        {
            "metric_name": "ROW_COUNT",
            "reconciliation_key": "KNOWLEDGE_ITEMS",
            "reconciliation_side": "TARGET",
            "metric_value_bigint": target_row_count
        }
    ]

    return (
        reconciliation_results,
        source_row_count,
        target_row_count
    )

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 15, Finished, Available, Finished, False)

## Prepare Gold Knowledge

Removes temporary validation flags from valid knowledge items and adds Gold execution lineage and audit metadata before promotion.

In [15]:
def prepare_gold_knowledge(
    valid_knowledge_items,
    execution_step_id,
    created_at,
    created_by
):
    """
    Removes temporary validation flags and adds Gold lineage
    and audit metadata.
    """

    validation_flag_columns = [
        column_name
        for column_name in valid_knowledge_items.columns
        if column_name.startswith("is_")
        and column_name.endswith("_passed")
    ]

    return (
        valid_knowledge_items
        .drop(
            *validation_flag_columns,
            "is_validation_passed"
        )
        .withColumn(
            "execution_step_id",
            F.lit(execution_step_id).cast("long")
        )
        .withColumn(
            "created_at",
            F.lit(created_at)
        )
        .withColumn(
            "created_by",
            F.lit(created_by)
        )
    )

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 17, Finished, Available, Finished, False)

## Load Valid Knowledge into Gold

Replaces the current trusted Gold knowledge snapshot with validated knowledge items.

In [16]:
def write_gold_knowledge(
    gold_knowledge_items
):
    """
    Replaces the current trusted Gold knowledge snapshot.
    """

    (
        gold_knowledge_items.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("gold.knowledge_items")
    )

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 18, Finished, Available, Finished, False)

## Determine Process Status

Evaluates validation and reconciliation results to determine whether the Gold validation process completed cleanly or requires observation.

In [ ]:
def determine_process_status(
    validation_results,
    reconciliation_results
):
    """
    Determines whether the process completed with validation
    or reconciliation observations.
    """

    has_validation_issues = any(
        result["affected_row_count"] > 0
        for result in validation_results
    )

    reconciliation_by_key = {}

    for result in reconciliation_results:
        key = result["reconciliation_key"]
        side = result["reconciliation_side"]

        reconciliation_by_key.setdefault(key, {})[side] = (
            result["metric_value_bigint"]
        )

    has_reconciliation_issues = any(
        values.get("SOURCE") != values.get("TARGET")
        for values in reconciliation_by_key.values()
    )

    return (
        has_validation_issues
        or has_reconciliation_issues
    )

## Execute Validation Process

Orchestrates Work loading, knowledge validation, reconciliation, Gold preparation, promotion, and final process status evaluation.

In [ ]:
created_at = datetime.now(timezone.utc)

validation_results = []

work_knowledge_items = load_work_knowledge()

work_knowledge_items = validate_required_knowledge_id(
    work_knowledge_items=work_knowledge_items,
    validation_results=validation_results
)

work_knowledge_items = validate_required_content(
    work_knowledge_items=work_knowledge_items,
    validation_results=validation_results
)

work_knowledge_items = validate_required_classification(
    work_knowledge_items=work_knowledge_items,
    validation_results=validation_results
)

work_knowledge_items = validate_knowledge_id_uniqueness(
    work_knowledge_items=work_knowledge_items,
    validation_results=validation_results
)

work_knowledge_items, valid_knowledge_items = (
    determine_validation_outcome(
        work_knowledge_items=work_knowledge_items
    )
)

(
    reconciliation_results,
    source_row_count,
    target_row_count
) = reconcile_knowledge_row_counts(
    work_knowledge_items=work_knowledge_items,
    valid_knowledge_items=valid_knowledge_items
)

gold_knowledge_items = prepare_gold_knowledge(
    valid_knowledge_items=valid_knowledge_items,
    execution_step_id=p_execution_step_id,
    created_at=created_at,
    created_by=p_created_by
)

write_gold_knowledge(
    gold_knowledge_items=gold_knowledge_items
)

has_observations = determine_process_status(
    validation_results=validation_results,
    reconciliation_results=reconciliation_results
)

## Output

Returns validation results, reconciliation metrics, final observation status, and summary record counts to the calling pipeline.

In [17]:
output = {
    "validation_results": validation_results,
    "reconciliation_results": reconciliation_results,
    "has_observations": has_observations,
    "records_validated": source_row_count,
    "records_promoted": target_row_count
}

mssparkutils.notebook.exit(
    json.dumps(output)
)

StatementMeta(, bce33e42-cdbd-4f0c-ab82-a09bb2430630, 19, Finished, Available, Finished, False)